# Kaggle Production Pipeline: TD3 + DDPG

Notebook flow:
1. Setup
2. Config
3. Batch selection
4. Robust training loop
5. Conditional plotting
6. Export

This notebook is fault-tolerant and resume-safe:
- completed experiments are skipped
- failed experiments are isolated
- plotting runs only after all 24 experiments complete

In [ ]:
import os
import sys
import glob
import json
import shutil
import subprocess
from pathlib import Path

# Clone repository if not already present
REPO_URL = "https://github.com/adityagangwani30/TD3-Car-Game.git"
REPO_DIR = "/kaggle/working/TD3-Car-Game"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("✓ Repository cloned successfully")
else:
    print("✓ Repository already exists")

# Change to repository directory
os.chdir(REPO_DIR)
print(f"✓ Working directory: {os.getcwd()}")

# Install dependencies
print("Installing dependencies from requirements.txt...")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("✓ Dependencies installed successfully")

# Check GPU availability
import torch
print("\n--- GPU Information ---")
print("GPU Available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# Create necessary directories
BASE_DIR = "/kaggle/working"
for folder in ["logs", "models", "results"]:
    Path(BASE_DIR, folder).mkdir(parents=True, exist_ok=True)
    Path(REPO_DIR, folder).mkdir(parents=True, exist_ok=True)

print("\n--- Directory Structure ---")
print(f"Logs:    {Path(BASE_DIR, 'logs')}")
print(f"Models:  {Path(BASE_DIR, 'models')}")
print(f"Results: {Path(BASE_DIR, 'results')}")

In [ ]:
from pathlib import Path
import json

from config import EXPERIMENTS, EXPERIMENT_REWARD_MODES, EXPERIMENT_SENSOR_NOISE_LEVELS

MAX_EPISODES = 5000
HEADLESS = True
ALGORITHMS = ["td3", "ddpg"]

assert len(EXPERIMENTS) == 12

def build_experiment_tags():
    tags = []
    for reward_mode in EXPERIMENT_REWARD_MODES:
        for noise in EXPERIMENT_SENSOR_NOISE_LEVELS:
            r_idx = EXPERIMENT_REWARD_MODES.index(reward_mode) + 1
            n_idx = EXPERIMENT_SENSOR_NOISE_LEVELS.index(noise) + 1
            tags.append(f"R{r_idx}_N{n_idx}")
    return tags

EXPERIMENT_TAGS = build_experiment_tags()

def _read_last_log_entry(log_file: Path):
    if not log_file.exists():
        return None
    last = None
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                last = json.loads(line)
            except json.JSONDecodeError:
                continue
    return last

def _last_logged_episode(algo, exp_id):
    log_file = Path(BASE_DIR, "logs", algo, exp_id, "training_log.jsonl")
    if not log_file.exists():
        return 0
    max_ep = 0
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                payload = json.loads(line)
            except json.JSONDecodeError:
                continue
            ep = int(payload.get("episode", 0) or 0)
            if ep > max_ep:
                max_ep = ep
    return max_ep

def is_experiment_complete(algo, exp_id, max_episodes=5000):
    model_dir = Path(BASE_DIR, "models", algo, exp_id)
    best_models = list(model_dir.glob("*_best.pth")) if model_dir.exists() else []
    if best_models:
        return True
    return _last_logged_episode(algo, exp_id) >= int(max_episodes)

def get_latest_checkpoint(algo, exp_id):
    model_dir = Path(BASE_DIR, "models", algo, exp_id)
    if not model_dir.exists():
        return None
    ckpts = []
    for p in model_dir.glob("*_ep*.pth"):
        stem = p.stem
        try:
            ep = int(stem.split("_ep")[-1])
        except ValueError:
            continue
        ckpts.append((ep, p))
    if not ckpts:
        return None
    ckpts.sort(key=lambda x: x[0], reverse=True)
    return ckpts[0][1]

def prune_experiment_models(algo, exp_id):
    model_dir = Path(BASE_DIR, "models", algo, exp_id)
    if not model_dir.exists():
        return

    best_candidates = sorted(model_dir.glob("*_best.pth"))
    if not best_candidates:
        best_candidates = sorted(model_dir.glob("*_best_avg100.pth"))
    best_keep = best_candidates[0] if best_candidates else None

    latest_ckpt = get_latest_checkpoint(algo, exp_id)

    for path in model_dir.glob("*.pth"):
        if best_keep and path == best_keep:
            continue
        if latest_ckpt and path == latest_ckpt:
            continue
        if path.name.endswith("_best_avg100.pth") and best_keep and path != best_keep:
            path.unlink(missing_ok=True)
            continue
        if "_ep" in path.stem:
            path.unlink(missing_ok=True)

def last_progress_line(algo, exp_id):
    log_file = Path(BASE_DIR, "logs", algo, exp_id, "training_log.jsonl")
    entry = _read_last_log_entry(log_file)
    if not entry:
        return "Episode 0/5000 | Reward: n/a | Avg100: n/a"
    ep = int(entry.get("episode", 0) or 0)
    reward = entry.get("reward_total", "n/a")
    avg100 = entry.get("reward_rolling_avg_100", "n/a")
    return f"Episode {ep}/{MAX_EPISODES} | Reward: {reward} | Avg100: {avg100}"

def all_experiments_done(max_episodes=MAX_EPISODES):
    for algo in ALGORITHMS:
        for exp_id in EXPERIMENT_TAGS:
            if not is_experiment_complete(algo, exp_id, max_episodes=max_episodes):
                return False
    return True

print("Total experiments:", len(ALGORITHMS) * len(EXPERIMENT_TAGS))

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R1_N1 (Basic, No Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 0 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R1_N2 (Basic, Low Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 1 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R1_N3 (Basic, High Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 2 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R2_N1 (Shaped, No Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 3 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R2_N2 (Shaped, Low Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 4 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R2_N3 (Shaped, High Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 5 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R3_N1 (Modified, No Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 6 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R3_N2 (Modified, Low Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 7 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R3_N3 (Modified, High Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 8 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R4_N1 (Tuned, No Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 9 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R4_N2 (Tuned, Low Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 10 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running TD3 -> R4_N3 (Tuned, High Noise)")
!python run_experiments.py \
    --algo td3 \
    --max-episodes 5000 \
    --start-index 11 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R1_N1 (Basic, No Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 0 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R1_N2 (Basic, Low Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 1 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R1_N3 (Basic, High Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 2 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R2_N1 (Shaped, No Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 3 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R2_N2 (Shaped, Low Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 4 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R2_N3 (Shaped, High Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 5 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R3_N1 (Modified, No Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 6 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R3_N2 (Modified, Low Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 7 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R3_N3 (Modified, High Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 8 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R4_N1 (Tuned, No Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 9 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R4_N2 (Tuned, Low Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 10 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
import os
os.chdir("/kaggle/working/TD3-Car-Game")
print("Running DDPG -> R4_N3 (Tuned, High Noise)")
!python run_experiments.py \
    --algo ddpg \
    --max-episodes 5000 \
    --start-index 11 \
    --max-experiments 1 \
    --headless \
    --resume

In [ ]:
if all_experiments_done():
    print("All 24 experiments completed. Running comparison plotting...")
    plot_cmd = f"""
python plot_metrics.py \
--compare-algos
"""
    !{plot_cmd}
    print("Plotting complete.")
else:
    print("Skipping plotting (incomplete runs)")

In [ ]:
Path(BASE_DIR, "results").mkdir(parents=True, exist_ok=True)
shutil.make_archive('/kaggle/working/final_results', 'zip', '/kaggle/working/results')
print('Saved: /kaggle/working/final_results.zip')